In [9]:
%pip install -q transnetv2-pytorch

In [19]:
import torch
from transnetv2_pytorch import TransNetV2
from pathlib import Path
import cv2
import os
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [11]:
# Initialize the model ('auto' selects CUDA, MPS, or CPU based on availability)
model = TransNetV2(device="auto")
model.eval()


video_path = str(Path("/content/drive/MyDrive/mock video/1.mp4"))
output_dir = Path("/content/drive/MyDrive/keyframes/")
output_dir.mkdir(parents=True, exist_ok=True)

# Method A: High-level scene detection (returns formatted timestamps and frame indices)
scenes = model.detect_scenes(video_path, threshold=0.5)

print(f"Found {len(scenes)} scenes:")

for scene in scenes:
    print(
        f"Shot {scene['shot_id']}: "
        f"Frames {scene['start_frame']}-{scene['end_frame']} "
        f"({scene['start_time']} to {scene['end_time']})"
    )

Found 336 scenes:
Shot 1: Frames 0-8 (0.000 to 0.267)
Shot 2: Frames 9-53 (0.300 to 1.767)
Shot 3: Frames 54-342 (1.800 to 11.400)
Shot 4: Frames 343-410 (11.433 to 13.667)
Shot 5: Frames 411-457 (13.700 to 15.233)
Shot 6: Frames 458-509 (15.267 to 16.967)
Shot 7: Frames 510-564 (17.000 to 18.800)
Shot 8: Frames 565-611 (18.833 to 20.367)
Shot 9: Frames 612-660 (20.400 to 22.000)
Shot 10: Frames 661-721 (22.033 to 24.033)
Shot 11: Frames 722-761 (24.067 to 25.367)
Shot 12: Frames 762-856 (25.400 to 28.533)
Shot 13: Frames 857-920 (28.567 to 30.667)
Shot 14: Frames 921-1886 (30.700 to 62.867)
Shot 15: Frames 1887-2035 (62.900 to 67.833)
Shot 16: Frames 2036-2130 (67.867 to 71.000)
Shot 17: Frames 2131-2216 (71.033 to 73.867)
Shot 18: Frames 2217-2287 (73.900 to 76.233)
Shot 19: Frames 2288-2374 (76.267 to 79.133)
Shot 20: Frames 2375-2411 (79.167 to 80.367)
Shot 21: Frames 2412-2574 (80.400 to 85.800)
Shot 22: Frames 2575-2719 (85.833 to 90.633)
Shot 23: Frames 2720-2815 (90.667 to 93.8

In [26]:
# 1. Initialize CLIP Model & Processor once
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

import torch.nn.functional as F

def get_clip_embeddings(frames):
    """Return L2-normalized CLIP image embeddings for BGR frames."""
    if not frames:
        return np.empty((0, model.config.projection_dim), dtype=np.float32)

    rgb_frames = [
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        for frame in frames
    ]

    inputs = processor(
        images=rgb_frames,
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"].to(device)

    with torch.inference_mode():
        outputs = model.get_image_features(
            pixel_values=pixel_values
        )

        # New Transformers versions return BaseModelOutputWithPooling.
        if hasattr(outputs, "pooler_output"):
            embeds = outputs.pooler_output
        else:
            # Older versions return the embedding tensor directly.
            embeds = outputs

        embeds = F.normalize(embeds, p=2, dim=-1)

    return embeds.cpu().numpy()

def extract_evenly_spaced_frames(video_path, start_idx, end_idx, num_frames=4):
    """Samples up to 4 evenly spaced frames from a scene range."""
    target_indices = np.unique(np.linspace(start_idx, end_idx, num_frames, dtype=int))

    cap = cv2.VideoCapture(video_path)
    frames, indices = [], []

    for idx in target_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
            indices.append(idx)

    cap.release()
    return frames, indices


def frame_filtering(video_path, start_idx, end_idx, threshold=0.9):
    """
    Implements Algorithm 1 (Frame Filtering):
    1. Extract evenly spaced candidate frames from the scene.
    2. Embed each candidate with CLIP.
    3. Greedily keep a frame only if it's NOT a near-duplicate
       (cosine similarity > threshold) of any frame already kept.
    """
    candidate_frames, candidate_indices = extract_evenly_spaced_frames(
        video_path, start_idx, end_idx, num_frames=4
    )

    if len(candidate_frames) == 0:
        return [], []

    embeddings = get_clip_embeddings(candidate_frames)

    kept_frames, kept_indices, kept_embeds = [], [], []

    for frame, idx, emb in zip(candidate_frames, candidate_indices, embeddings):
        is_duplicate = False

        for kept_emb in kept_embeds:
            # embeddings are already normalized -> dot product = cosine similarity
            sim = float(np.dot(emb, kept_emb))
            if sim > threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            kept_frames.append(frame)
            kept_indices.append(idx)
            kept_embeds.append(emb)

    return kept_frames, kept_indices



# Loop over scenes detected by TransNetV2
# Ensure video_path is correctly set before processing frames
video_path = str(Path("/content/drive/MyDrive/mock video/1.mp4")) # Re-assign the correct path

for scene in scenes:
    shot_id = scene['shot_id']
    start_idx = scene['start_frame']
    end_idx = scene['end_frame']

    print(f"Processing Shot #{shot_id} (Frames {start_idx}-{end_idx})...")

    # Get filtered frames and their original frame numbers
    kept_frames, kept_indices = frame_filtering(video_path, start_idx, end_idx, threshold=0.9)

    # Save kept frames directly to the folder
    for frame, frame_idx in zip(kept_frames, kept_indices):
        file_name = f"shot_{shot_id:03d}_frame_{frame_idx:06d}.jpg"
        save_path = os.path.join(output_dir, file_name)

        cv2.imwrite(save_path, frame)
        print(f" Saved: {save_path}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Processing Shot #1 (Frames 0-8)...
 Saved: /content/drive/MyDrive/keyframes/shot_001_frame_000000.jpg
 Saved: /content/drive/MyDrive/keyframes/shot_001_frame_000008.jpg
Processing Shot #2 (Frames 9-53)...
 Saved: /content/drive/MyDrive/keyframes/shot_002_frame_000009.jpg
 Saved: /content/drive/MyDrive/keyframes/shot_002_frame_000023.jpg
 Saved: /content/drive/MyDrive/keyframes/shot_002_frame_000053.jpg
Processing Shot #3 (Frames 54-342)...
 Saved: /content/drive/MyDrive/keyframes/shot_003_frame_000054.jpg
 Saved: /content/drive/MyDrive/keyframes/shot_003_frame_000150.jpg
Processing Shot #4 (Frames 343-410)...
 Saved: /content/drive/MyDrive/keyframes/shot_004_frame_000343.jpg
 Saved: /content/drive/MyDrive/keyframes/shot_004_frame_000365.jpg
 Saved: /content/drive/MyDrive/keyframes/shot_004_frame_000387.jpg
Processing Shot #5 (Frames 411-457)...
 Saved: /content/drive/MyDrive/keyframes/shot_005_frame_000411.jpg
 Saved: /content/drive/MyDrive/keyframes/shot_005_frame_000426.jpg
Processin

In [27]:
import glob

# Match all .jpg files in that directory
jpg_counter = len(glob.glob1(output_dir, "*.jpg"))

print(f"Total JPG files: {jpg_counter}")

Total JPG files: 687
